# CiteScope — Modelado · Fase 4b: Fine-tuning de SciBERT

**Objetivo.** Explorar si un modelo transformer preentrenado en texto científico (**SciBERT**, `allenai/scibert_scivocab_uncased`) supera de forma clara al mejor clásico (LogReg enriquecido, Macro F1 ≈ 0,628 en val) y se acerca a la meta de 0,70.

**Entregables de esta fase.**
- Fine-tuning de SciBERT sobre `text_enriched` para clasificación en 8 subáreas.
- Evaluación en **validación** (Macro F1, accuracy, F1 por clase) comparable con el clásico.
- Resultados registrados en `models/artifacts/scibert_results.csv`.

> Mismo split anti-fuga (Fase 1) y semilla. Entrena en `train`, evalúa en `val`; el **test sigue intacto**. Requiere `torch`, `transformers`, `datasets`, `accelerate` y descarga del modelo (~440 MB). En Apple Silicon usa **MPS**.


## Entorno y dispositivo

Verificamos que `torch` y `transformers` estén disponibles y seleccionamos el dispositivo (MPS en Apple Silicon, si no CPU).


In [1]:
import os
import random

# El CDN Xet (us.aws.cdn.hf.co) puede estar bloqueado por el proxy: forzar descarga HTTP clásica.
os.environ["HF_HUB_DISABLE_XET"] = "1"

import numpy as np
import torch
import transformers

# Redes con inspección TLS: usar el llavero del SO (truststore) confía en la CA corporativa.
try:
    import truststore
    truststore.inject_into_ssl()
    ssl_mode = "truststore (llavero del SO)"
except ImportError:
    import certifi
    os.environ["SSL_CERT_FILE"] = certifi.where()
    os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
    os.environ["CURL_CA_BUNDLE"] = certifi.where()
    ssl_mode = "certifi (instala 'truststore' si hay inspección TLS)"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

print(f"torch:        {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"SSL:          {ssl_mode}")
print(f"Xet:          desactivado (HF_HUB_DISABLE_XET=1)")
print(f"Dispositivo:  {device}")


torch:        2.13.0
transformers: 5.16.1
SSL:          truststore (llavero del SO)
Xet:          desactivado (HF_HUB_DISABLE_XET=1)
Dispositivo:  mps


## Carga de datos, split y etiquetas

Cargamos el dataset, construimos `text_enriched`, adjuntamos el split de la Fase 1 y codificamos las 8 subáreas a enteros (`label2id`).


In [2]:
from pathlib import Path

import pandas as pd

TARGET = "citing_primary_category"
BASELINE_ENRICHED_VAL = 0.6277  # mejor clásico (Fase 3)

REPO_ROOT = Path.cwd().parent
DATASET_DIR = REPO_ROOT / "Dataset"
ARTIFACTS_DIR = Path.cwd() / "artifacts"

CANONICAL = DATASET_DIR / "unarxive_microproyecto.jsonl"
LOCAL_COPY = DATASET_DIR / "copy_unarxive_microproyecto.jsonl"
DATA_PATH = CANONICAL if CANONICAL.exists() else LOCAL_COPY

df = pd.read_json(DATA_PATH, lines=True, dtype={"citing_arxiv_id": "string"})
ctx = df["citation_context"].fillna("").astype(str).str.strip()
title = df["cited_title"].fillna("").astype(str).str.strip()
abstract = df["cited_abstract"].fillna("").astype(str).str.strip()
df["text_enriched"] = ["\n".join(p for p in (c, t, a) if p) for c, t, a in zip(ctx, title, abstract)]

split_map = pd.read_csv(ARTIFACTS_DIR / "split_assignment.csv")[["citation_id", "split"]]
df = df.merge(split_map, on="citation_id", how="left")

labels = sorted(df[TARGET].unique())
label2id = {c: i for i, c in enumerate(labels)}
id2label = {i: c for c, i in label2id.items()}
df["label"] = df[TARGET].map(label2id)

train = df[df["split"] == "train"]
val = df[df["split"] == "val"]

print(f"train: {len(train)} | val: {len(val)} | clases: {len(labels)}")
print("label2id:", label2id)


train: 2400 | val: 800 | clases: 8
label2id: {'cs.AI': 0, 'cs.CL': 1, 'cs.CV': 2, 'cs.IR': 3, 'cs.LG': 4, 'cs.MA': 5, 'cs.NE': 6, 'cs.RO': 7}


## Tokenización

Tokenizamos con el tokenizer de SciBERT, truncando a **512 tokens** (el máximo de SciBERT) para cubrir el abstract completo del citado, que antes se truncaba a 256. El padding es dinámico por lote (`DataCollatorWithPadding`).


In [3]:
from transformers import AutoTokenizer, DataCollatorWithPadding

MODEL_NAME = "allenai/scibert_scivocab_uncased"
MAX_LEN = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class TextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.enc = tokenizer(list(texts), truncation=True, max_length=MAX_LEN)
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item


train_ds = TextDataset(train["text_enriched"], train["label"])
val_ds = TextDataset(val["text_enriched"], val["label"])
collator = DataCollatorWithPadding(tokenizer)

print(f"Tokenizado. train={len(train_ds)} val={len(val_ds)} | max_len={MAX_LEN}")


Tokenizado. train=2400 val=800 | max_len=512


## Modelo y entrenamiento

Fine-tuning de SciBERT con cabeza de clasificación (8 clases). Ahora con `max_len=512`, hasta 5 épocas con **early stopping** y `load_best_model_at_end` para conservar la mejor época por **Macro F1**. Batch 8 para evitar falta de memoria en MPS con secuencias largas. Esta celda es la más lenta (bastante más que con 256).


In [4]:
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(labels), id2label=id2label, label2id=label2id,
)


def compute_metrics(eval_pred):
    logits, y_true = eval_pred
    y_pred = logits.argmax(axis=-1)
    return {
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "accuracy": accuracy_score(y_true, y_pred),
    }


args = TrainingArguments(
    output_dir=str(ARTIFACTS_DIR / "scibert_ckpt"),
    num_train_epochs=5,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=50,
    seed=SEED,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those

Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,1.013461,0.975534,0.672489,0.673750
2,0.713845,0.972902,0.675587,0.672500
3,0.494873,1.023848,0.683597,0.680000
4,0.266919,1.271752,0.668151,0.666250
5,0.159771,1.318867,0.667827,0.663750


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/germanrodriguez/Desktop/Code/CSCO/MAIA4401_MicroProyecto/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


...

/Users/germanrodriguez/Desktop/Code/CSCO/MAIA4401_MicroProyecto/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/germanrodriguez/Desktop/Code/CSCO/MAIA4401_MicroProyecto/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1500, training_loss=0.5955406681696573, metrics={'train_runtime': 2871.3057, 'train_samples_per_second': 4.179, 'train_steps_per_second': 0.522, 'total_flos': 2992856253745536.0, 'train_loss': 0.5955406681696573, 'epoch': 5.0})

## Evaluación en validación

Comparamos SciBERT contra el mejor clásico (0,628) y revisamos el reporte por clase para ver si mejora las categorías difíciles (`cs.LG`, `cs.AI`).


In [5]:
from sklearn.metrics import classification_report

val_logits = trainer.predict(val_ds).predictions
val_pred = val_logits.argmax(axis=-1)
y_true = val["label"].to_numpy()

macro_f1_val = f1_score(y_true, val_pred, average="macro")
acc_val = accuracy_score(y_true, val_pred)

print(f"Clásico enriquecido (val): Macro F1 = {BASELINE_ENRICHED_VAL:.4f}")
print(f"SciBERT (val):             Macro F1 = {macro_f1_val:.4f}  (delta {macro_f1_val - BASELINE_ENRICHED_VAL:+.4f})")
print(f"SciBERT (val):             Accuracy = {acc_val:.4f}")
print("\nReporte por clase (SciBERT, val):\n")
print(classification_report(y_true, val_pred, target_names=labels, digits=3))


/Users/germanrodriguez/Desktop/Code/CSCO/MAIA4401_MicroProyecto/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Clásico enriquecido (val): Macro F1 = 0.6277
SciBERT (val):             Macro F1 = 0.6836  (delta +0.0559)
SciBERT (val):             Accuracy = 0.6800

Reporte por clase (SciBERT, val):

              precision    recall  f1-score   support

       cs.AI      0.578     0.520     0.547       100
       cs.CL      0.745     0.760     0.752       100
       cs.CV      0.728     0.750     0.739       100
       cs.IR      0.692     0.740     0.715       100
       cs.LG      0.398     0.490     0.439       100
       cs.MA      0.774     0.720     0.746       100
       cs.NE      0.768     0.730     0.749       100
       cs.RO      0.839     0.730     0.781       100

    accuracy                          0.680       800
   macro avg      0.690     0.680     0.684       800
weighted avg      0.690     0.680     0.684       800



## Registro de resultados

Guardamos las métricas de SciBERT en `models/artifacts/scibert_results.csv`.


In [8]:
ARTIFACTS_DIR.mkdir(exist_ok=True)

scibert_row = {
    "modelo": "scibert", "input": "text_enriched",
    "macro_f1_val": round(macro_f1_val, 4),
    "accuracy_val": round(acc_val, 4),
    "delta_vs_clasico": round(macro_f1_val - BASELINE_ENRICHED_VAL, 4),
    "model_name": MODEL_NAME, "max_len": MAX_LEN, "max_epochs": 5,
    "early_stopping": True, "lr": 2e-5, "seed": SEED,
}
scibert_df = pd.DataFrame([scibert_row])
scibert_df.to_csv(ARTIFACTS_DIR / "scibert_results.csv", index=False)

print("Guardado en:", (ARTIFACTS_DIR / "scibert_results.csv").relative_to(REPO_ROOT))
scibert_df


Guardado en: models/artifacts/scibert_results.csv


,modelo,input,macro_f1_val,accuracy_val,delta_vs_clasico,model_name,max_len,max_epochs,early_stopping,lr,seed
0,scibert,text_enriched,0.6836,0.68,0.0559,allenai/scibert_scivocab_uncased,512,5,True,0.00002,42
